# Population Genetics and the Mathematics of Inheritance Workflow

This notebook scaffold supports the article **Population Genetics and the Mathematics of Inheritance**. It can be expanded with Hardy-Weinberg expectations, genotype-frequency calculation, expected heterozygosity, selection, mutation, migration, drift, population structure, migration-selection balance, and provenance documentation.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

article_dir = Path.cwd().parent
scenarios = pd.read_csv(article_dir / 'data' / 'population_genetics_scenarios.csv')
scenarios['q0'] = 1 - scenarios['p0']
scenarios['expected_AA'] = scenarios['p0'] ** 2
scenarios['expected_Aa'] = 2 * scenarios['p0'] * scenarios['q0']
scenarios['expected_aa'] = scenarios['q0'] ** 2
scenarios[['scenario', 'p0', 'q0', 'expected_AA', 'expected_Aa', 'expected_aa']].round(4)

In [ ]:
genotypes = pd.read_csv(article_dir / 'data' / 'genotype_matrix.csv')
locus_cols = [c for c in genotypes.columns if c.startswith('locus_')]
summaries = []

for locus in locus_cols:
    values = genotypes[locus].to_numpy()
    n = len(values)
    n_AA = np.sum(values == 2)
    n_Aa = np.sum(values == 1)
    p = (2 * n_AA + n_Aa) / (2 * n)
    summaries.append({'locus': locus, 'n': n, 'p': p, 'He': 2 * p * (1 - p)})

pd.DataFrame(summaries).round(4)

In [ ]:
freqs = pd.read_csv(article_dir / 'data' / 'multipop_allele_frequencies.csv')
pop_cols = [c for c in freqs.columns if c.startswith('pop')]
freqs['pbar'] = freqs[pop_cols].mean(axis=1)
freqs['HS'] = (2 * freqs[pop_cols] * (1 - freqs[pop_cols])).mean(axis=1)
freqs['HT'] = 2 * freqs['pbar'] * (1 - freqs['pbar'])
freqs['fst_style'] = np.where(freqs['HT'] > 0, (freqs['HT'] - freqs['HS']) / freqs['HT'], 0)
freqs[['locus', 'pbar', 'HS', 'HT', 'fst_style']].round(4)

In [ ]:
condition = pd.read_csv(article_dir / 'data' / 'population_condition_sites.csv')
condition['population_condition_score'] = (
    0.18 * condition['heterozygosity'] +
    0.18 * condition['allelic_richness'] +
    0.16 * condition['gene_flow'] +
    0.16 * (1 - condition['fragmentation_pressure']) +
    0.16 * (1 - condition['bottleneck_risk']) +
    0.16 * condition['adaptive_capacity']
)
condition.sort_values('population_condition_score', ascending=False).round(3)